Imports


In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

from scipy import sparse

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42

Charger les données

In [2]:
DATA_PATH = Path("../data/raw/bs140513_032310.csv")

df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
df.head()

Shape: (594643, 10)


,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud
0,0,'C1093826151','4','M','28007','M348934600','28007','es_transportation',4.55,0
1,0,'C352968107','2','M','28007','M348934600','28007','es_transportation',39.68,0
2,0,'C2054744914','4','F','28007','M1823072687','28007','es_transportation',26.89,0
3,0,'C1760612790','3','M','28007','M348934600','28007','es_transportation',17.25,0
4,0,'C757503768','5','M','28007','M348934600','28007','es_transportation',35.72,0


Nettoyer les chaînes

In [3]:
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    df[col] = df[col].str.strip("'")

df.head()

C:\Users\bench\AppData\Local\Temp\ipykernel_19464\2971451087.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns


,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud
0,0,C1093826151,4,M,28007,M348934600,28007,es_transportation,4.55,0
1,0,C352968107,2,M,28007,M348934600,28007,es_transportation,39.68,0
2,0,C2054744914,4,F,28007,M1823072687,28007,es_transportation,26.89,0
3,0,C1760612790,3,M,28007,M348934600,28007,es_transportation,17.25,0
4,0,C757503768,5,M,28007,M348934600,28007,es_transportation,35.72,0


Nettoyer les chaînes

In [4]:
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    df[col] = df[col].str.strip("'")

df.head()

C:\Users\bench\AppData\Local\Temp\ipykernel_19464\2971451087.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns


,step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud
0,0,C1093826151,4,M,28007,M348934600,28007,es_transportation,4.55,0
1,0,C352968107,2,M,28007,M348934600,28007,es_transportation,39.68,0
2,0,C2054744914,4,F,28007,M1823072687,28007,es_transportation,26.89,0
3,0,C1760612790,3,M,28007,M348934600,28007,es_transportation,17.25,0
4,0,C757503768,5,M,28007,M348934600,28007,es_transportation,35.72,0


Créer amount_log

In [5]:
df["amount_log"] = np.log1p(df["amount"])

df[["amount", "amount_log"]].head(10)

,amount,amount_log
0,4.55,1.713798
1,39.68,3.705737
2,26.89,3.328268
3,17.25,2.904165
4,35.72,3.603322
5,25.81,3.288775
6,9.10,2.312535
7,21.17,3.098740
8,32.40,3.508556
9,35.40,3.594569


Faire un split temporel

In [6]:
train_df = df[df["step"] <= 119].copy()

valid_df = df[
    (df["step"] >= 120) &
    (df["step"] <= 149)
].copy()

test_df = df[df["step"] >= 150].copy()

print("Train :", train_df.shape)
print("Valid :", valid_df.shape)
print("Test  :", test_df.shape)

Train : (374914, 11)
Valid : (108423, 11)
Test  : (111306, 11)


In [7]:
print(
    "Train steps:",
    train_df["step"].min(),
    "→",
    train_df["step"].max()
)

print(
    "Validation steps:",
    valid_df["step"].min(),
    "→",
    valid_df["step"].max()
)

print(
    "Test steps:",
    test_df["step"].min(),
    "→",
    test_df["step"].max()
)

Train steps: 0 → 119
Validation steps: 120 → 149
Test steps: 150 → 179


In [8]:
def split_summary(data, name):
    return {
        "split": name,
        "rows": len(data),
        "frauds": int(data["fraud"].sum()),
        "fraud_rate": data["fraud"].mean()
    }


summary = pd.DataFrame([
    split_summary(train_df, "train"),
    split_summary(valid_df, "validation"),
    split_summary(test_df, "test")
])

summary

,split,rows,frauds,fraud_rate
0,train,374914,4800,0.012803
1,validation,108423,1200,0.011068
2,test,111306,1200,0.010781


In [9]:
numeric_features = [
    "step",
    "amount_log"
]

categorical_features = [
    "age",
    "gender",
    "merchant",
    "category"
]

model_features = numeric_features + categorical_features

model_features

['step', 'amount_log', 'age', 'gender', 'merchant', 'category']

Séparer X et y

In [10]:
X_train_raw = train_df[model_features].copy()
X_valid_raw = valid_df[model_features].copy()
X_test_raw = test_df[model_features].copy()

y_train = train_df["fraud"].copy()
y_valid = valid_df["fraud"].copy()
y_test = test_df["fraud"].copy()

In [11]:
X_train_raw.head()

,step,amount_log,age,gender,merchant,category
0,0,1.713798,4,M,M348934600,es_transportation
1,0,3.705737,2,M,M348934600,es_transportation
2,0,3.328268,4,F,M1823072687,es_transportation
3,0,2.904165,3,M,M348934600,es_transportation
4,0,3.603322,5,M,M348934600,es_transportation


In [12]:
numeric_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [13]:
X_train = preprocessor.fit_transform(X_train_raw)

X_valid = preprocessor.transform(X_valid_raw)

X_test = preprocessor.transform(X_test_raw)

In [14]:
print("Raw train :", X_train_raw.shape)
print("Processed :", X_train.shape)

print()
print("Raw valid :", X_valid_raw.shape)
print("Processed :", X_valid.shape)

print()
print("Raw test :", X_test_raw.shape)
print("Processed :", X_test.shape)

Raw train : (374914, 6)
Processed : (374914, 79)

Raw valid : (108423, 6)
Processed : (108423, 79)

Raw test : (111306, 6)
Processed : (111306, 79)


In [15]:
feature_names = preprocessor.get_feature_names_out()

print(f"Number of features: {len(feature_names)}")

feature_names

Number of features: 79


array(['num__step', 'num__amount_log', 'cat__age_0', 'cat__age_1',
       'cat__age_2', 'cat__age_3', 'cat__age_4', 'cat__age_5',
       'cat__age_6', 'cat__age_U', 'cat__gender_E', 'cat__gender_F',
       'cat__gender_M', 'cat__gender_U', 'cat__merchant_M1053599405',
       'cat__merchant_M117188757', 'cat__merchant_M1198415165',
       'cat__merchant_M1294758098', 'cat__merchant_M1313686961',
       'cat__merchant_M1352454843', 'cat__merchant_M1353266412',
       'cat__merchant_M1400236507', 'cat__merchant_M1416436880',
       'cat__merchant_M151143676', 'cat__merchant_M1535107174',
       'cat__merchant_M1600850729', 'cat__merchant_M1649169323',
       'cat__merchant_M1726401631', 'cat__merchant_M17379832',
       'cat__merchant_M1741626453', 'cat__merchant_M1748431652',
       'cat__merchant_M1788569036', 'cat__merchant_M1823072687',
       'cat__merchant_M1842530320', 'cat__merchant_M1872033263',
       'cat__merchant_M1873032707', 'cat__merchant_M1888755466',
       'cat__merchan

In [16]:
assert X_train.shape[1] == X_valid.shape[1]
assert X_train.shape[1] == X_test.shape[1]

assert "fraud" not in feature_names
assert "customer" not in feature_names

print("Preprocessing checks passed.")

Preprocessing checks passed.


In [17]:
MODELS_PATH = Path("../models")
MODELS_PATH.mkdir(parents=True, exist_ok=True)

joblib.dump(
    preprocessor,
    MODELS_PATH / "preprocessor.joblib"
)

print("Preprocessor saved.")

Preprocessor saved.


In [18]:
feature_names_list = feature_names.tolist()

with open(
    MODELS_PATH / "feature_names.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        feature_names_list,
        f,
        indent=2
    )

In [19]:
PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

sparse.save_npz(
    PROCESSED_PATH / "X_train.npz",
    sparse.csr_matrix(X_train)
)

sparse.save_npz(
    PROCESSED_PATH / "X_valid.npz",
    sparse.csr_matrix(X_valid)
)

sparse.save_npz(
    PROCESSED_PATH / "X_test.npz",
    sparse.csr_matrix(X_test)
)

np.save(
    PROCESSED_PATH / "y_train.npy",
    y_train.to_numpy()
)

np.save(
    PROCESSED_PATH / "y_valid.npy",
    y_valid.to_numpy()
)

np.save(
    PROCESSED_PATH / "y_test.npy",
    y_test.to_numpy()
)

print("Processed datasets saved.")

Processed datasets saved.
